# Machine Learning with Scikit-Learn

**Machine learning** is teaching a computer to find patterns in data — without being explicitly programmed for every case. Given enough examples, the model *learns* the relationship between inputs (features) and outputs (targets), then predicts outputs for data it has never seen.

**Scikit-learn** (`sklearn`) is Python's most popular ML library: simple, consistent, and packed with ready-made algorithms.

**What you will learn:**

- The supervised-learning workflow: **split → train → predict → evaluate**
- Linear regression — the "hello world" of ML
- Reading model results (`coef_`, `intercept_`)
- Evaluation metrics: MSE and R²
- Running the full pipeline on a real dataset (diabetes)

> **Setup:** `pip install scikit-learn matplotlib`

## A Very Short ML Theory

| Term | Meaning |
|------|---------|
| **Features (`X`)** | The inputs the model learns from (e.g. hours studied) |
| **Target (`y`)** | What we want to predict (e.g. exam score) |
| **Training set** | The data the model *learns from* |
| **Test set** | Held-out data the model is *evaluated on* (it never saw it) |
| **Supervised learning** | We have labeled examples (features + known answers) |
| **Unsupervised learning** | No labels — the model finds structure (clusters) on its own |

Splitting into train/test is critical: evaluating on the training data is cheating — the model already memorized it. That's why we always hold out a test set.

## Setup

Check that scikit-learn is installed and see the version:

In [ ]:
import sklearn
print(f"Scikit-learn version: {sklearn.__version__}")

## The Five-Step ML Workflow

Every supervised-learning project follows the same loop:

1. **Prepare data** — features `X`, target `y`
2. **Split** — `train_test_split()` holds out a test set
3. **Train** — `model.fit(X_train, y_train)` learns from the examples
4. **Predict** — `model.predict(X_test)` answers for unseen data
5. **Evaluate** — compare predictions vs. real answers (`MSE`, `R²`)

## Step 1 & 2: Prepare and Split the Data

Our toy problem: `y = 5 × X` — a perfectly linear relationship. We build it ourselves so we can verify the model finds the true pattern.

> **Shape matters:** scikit-learn wants `X` as a 2D array (rows = samples, columns = features), so a single feature still needs `[[1], [2], ...]` — that's the `[:, None]` trick.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Feature: numbers 1..10.  Target: exactly 5 * X (a perfect line)
X = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])
y = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50])

# Hold out 20% of the data for testing (random_state makes it reproducible)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training samples:", X_train.shape[0], "| Test samples:", X_test.shape[0])
print("Test features:  ", X_test.ravel())
print("Test answers:   ", y_test)

## Step 3 & 4: Train the Model and Predict

`LinearRegression` assumes the target is a **linear combination** of the features:

```
y = coef_ × x + intercept_
```

`model.fit()` finds the coefficients that best fit the training data. On our toy data it should recover *approximately* `coef_ ≈ 5` and `intercept_ ≈ 0` — the formula we generated the data with!

In [ ]:
from sklearn.linear_model import LinearRegression

# Create the model, train it on the training set
model = LinearRegression()
model.fit(X_train, y_train)

# The learned formula
print(f"Learned formula:  y = {model.coef_[0]:.4f} * x + {model.intercept_:.4f}")

# Predict the test answers (the model has never seen these rows!)
y_pred = model.predict(X_test)
print("Predicted values:", y_pred)
print("Actual values:   ", y_test)

## Step 5: Evaluate the Model

Metrics quantify *how wrong* the predictions are:

| Metric | Meaning | Perfect score |
|--------|---------|---------------|
| **MSE** | Average squared error (penalizes big mistakes) | 0 |
| **R²** | Fraction of variance explained by the model | 1.0 (100%) |

On perfect linear data, the model should hit **MSE ≈ 0** and **R² ≈ 1**.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared Score (R²):     {r2:.4f}")

### Visualizing the Result

Plotting makes the fit obvious — the points are the data, the line is the model. Perfect linear data → perfect fit.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.scatter(X, y, color="steelblue", label="Data points")
plt.plot(X, model.predict(X), color="crimson", linewidth=2, label="Model line")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Linear Regression Fit (y = 5x)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## A Real Dataset: Diabetes Progression

Toy data is nice, but let's run the *exact same* workflow on a **real** dataset: scikit-learn ships a built-in diabetes dataset (442 patients, 10 medical features, target = disease progression after one year). The code is identical — that's the power of scikit-learn's consistent API.

> **Note:** real data is never perfect — the R² here will be far from 1.

In [ ]:
from sklearn.datasets import load_diabetes

# Load the built-in dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
print("Features:", diabetes.feature_names)
print("Shape of X:", X.shape, "| Shape of y:", y.shape)

In [ ]:
# The same 5-step workflow, on the real data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"R²  on real data: {r2_score(y_test, y_pred):.3f}")
print(f"MSE on real data: {mean_squared_error(y_test, y_pred):.1f}")

# Which medical feature influences progression the most?
feature_importance = sorted(zip(diabetes.feature_names, model.coef_), key=lambda t: -abs(t[1]))
print("\nStrongest features (by |coefficient|):")
for name, coef in feature_importance[:3]:
    print(f"  {name:8s} -> {coef:+.1f}")

## 🎯 Key Takeaways

- ML workflow: **prepare → split → train → predict → evaluate**.
- `train_test_split(X, y, test_size=0.2, random_state=42)` holds out test data reproducibly.
- `LinearRegression().fit(X_train, y_train)` learns `y = coef_ × x + intercept_`.
- `predict()` answers for data the model has never seen.
- **MSE** (0 = perfect) and **R²** (1 = perfect) quantify the fit.
- scikit-learn's API is identical across models — swap `LinearRegression()` for another algorithm and the rest stays the same.
- Real data is messy: expect modest R² and always check which features matter.

## 🏋️ Practice Exercises

1. Add noise: `y = 5*X + np.random.normal(0, 2, size=X.shape)` — how much do MSE and R² change?
2. Try `test_size=0.5` — what happens to the metrics (and why)?
3. Replace `LinearRegression()` with `DecisionTreeRegressor()` (from `sklearn.tree`) — same workflow, same metrics. Compare the two.
4. Print predictions vs. actual values side by side for the diabetes test set (first 5 rows).
5. Use `model.predict([[100]])` on the toy model — what answer do you expect?

## 🚀 Next Steps

- **`09_Matplotlib_Visualization.ipynb`** — more ways to visualize data and model results.
- **`03_Pandas_Data_Analysis`** — clean and explore real datasets before modeling.
- Try **classification** next: `sklearn.neighbors.KNeighborsClassifier` on the iris dataset is the classic follow-up.